In [1]:
import numpy as np
import pandas as pd
import copy

import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

from data import fetch_dataset
from util import move_sliding_window, num_params
from model import LSTMModel
from algorithm import poc

pd.set_option('display.max_columns', None)
pd.set_option('display.max_columns', None)
np.set_printoptions(suppress=True, floatmode='fixed')

C:\ProgramData\Miniconda3\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('running on gpu')
else:
    device = torch.device("cpu")

running on gpu


# Parameters

In [3]:
window_size = 90 # Define window_size period and split inputs/labels\
batch_size = 1024
label_col_index = 0

# seq_len = 90  # (timestamps)
hidden_dim = 50
n_layers = 2
lr = 0.0001
output_dim = 1

#fed train params
num_local_epochs = 1
max_rounds = 100 #nb of total rounds for training
num_clients_per_round = 20 #number of clients to participate in training

# Data Preperation

In [4]:
dataframes = fetch_dataset("./cleaned")

Floor0
Floor1
Floor10
Floor11
Floor12
Floor13
Floor14
Floor15
Floor16
Floor17_p10
Floor18_p7
Floor19_p8
Floor2
Floor20
Floor21
Floor22
Floor23
Floor24
Floor25
Floor26
Floor27_p2
Floor28_p4
Floor29_p3
Floor3
Floor30
Floor31
Floor32
Floor33
Floor34
Floor35
Floor36
Floor37_p7
Floor38_p5
Floor39_p11
Floor4
Floor40
Floor41
Floor42
Floor43
Floor44
Floor45
Floor46
Floor47_p7
Floor48_p11
Floor49_p3
Floor5
Floor50
Floor51
Floor52
Floor53
Floor54
Floor55
Floor56
Floor57_p7
Floor58_p7
Floor59_p2
Floor6
Floor60
Floor61
Floor62
Floor63
Floor64
Floor65
Floor66
Floor67_p2
Floor68_p7
Floor69_p8
Floor70
Floor71
Floor72
Floor73
Floor74
Floor75
Floor76
Floor77_p5
Floor78_p2
Floor79_p6
Floor7_p6
Floor80
Floor81
Floor82
Floor83
Floor84
Floor85
Floor86
Floor87_p3
Floor88_p4
Floor89_p6
Floor8_p6
Floor9_p7


In [5]:
n_clients = len(dataframes) #total number of clients to partition data into

total_data_points = sum(df.shape[0] for df in dataframes.values()) # Compute the total number of data points

# Compute the data ratios
data_ratios = [df.shape[0] / total_data_points for client_id, df in dataframes.items()]

In [6]:
for _, df in dataframes.items():
    print(df.columns)
    print(df.shape)

Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
      dtype='object')
(72864, 9)
Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
      dtype='object')
(72864, 9)
Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
      dtype='object')
(72864, 9)
Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
      dtype='object')
(72864, 9)
Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
      dtype='object')
(72864, 9)
Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
      dtype='object')
(72864, 9)
Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
      dtype='object')
(72864, 9)
Index(['total_demand', 'AC', 'Light', 'Plug', 'Year', 'Month', 'Day', 'Hour',
       'Minute'],
 

# Build the training set

In [7]:
train_loader = []
test_loader = []
label_scalers = []
for _, df in dataframes.items():
    inputs_cols_indices = range(0, df.shape[1])  # use (total_demand,Year,Month,Day,Hour,Minute) columns as features
    #move the window
    inputs, labels = move_sliding_window(
        df.values,
        window_size,
        inputs_cols_indices=inputs_cols_indices,
        label_col_index=label_col_index
    )

    # Normalize the input data columns
    sc = MinMaxScaler()
    # Obtaining the scaler for the labels(usage data) so that output can be re-scaled to actual value during evaluation
    label_sc = MinMaxScaler()

    # Split data into train/test portions and combining all data into a single array
    test_portion = int(0.2 * len(inputs))

    train_x = sc.fit_transform(inputs[:-test_portion].reshape(-1, window_size * df.shape[1]))
    train_x = train_x.reshape(-1, window_size, df.shape[1])
    train_y = label_sc.fit_transform(labels[:-test_portion])

    test_x = sc.transform(inputs[-test_portion:].reshape(-1, window_size * df.shape[1]))
    test_x = test_x.reshape(-1, window_size, df.shape[1])
    test_y = label_sc.transform(labels[-test_portion:])

    # test_x.append(testx)
    # test_y.append(testy)
    label_scalers.append(label_sc)

    # pytorch data loaders
    train_data = TensorDataset(torch.from_numpy(train_x).to('cpu'), torch.from_numpy(train_y).to('cpu'))
    train_loader.append(DataLoader(train_data, batch_size=batch_size, drop_last=True))# Drop the last incomplete batch
    test_data = TensorDataset(torch.from_numpy(test_x).to('cpu'), torch.from_numpy(test_y).to('cpu'))
    test_loader.append(DataLoader(test_data, batch_size=batch_size))# Drop the last incomplete batch

    # release some memory
    del train_x, train_y
input_dim = next(iter(train_loader[0]))[0].shape[2]  # 22

(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 9) (72774, 1)
(72774, 90, 

# LSTM model

In [8]:
#         self.global_parameter = torch.zeros(self.num_classes, self.dim)
        
#         # defining the model: here, logistic regression
#         self.model = torch.nn.Linear(self.dim, self.num_classes, bias=False)
        
#         # defining loss function: here, cross-entropy
#         self.criterion = torch.nn.CrossEntropyLoss()

#         # defining the optimizer: here, SGD
#         self.optimizer = torch.optim.SGD(self.model.parameters(), lr=self.lr, weight_decay=10e-4)

In [9]:
lstm = LSTMModel(input_dim, hidden_dim, output_dim, n_layers)
model_type = 'LSTM'
print(lstm)
print(num_params(lstm))

LSTMModel(
  (lstm): LSTM(9, 50, num_layers=2, batch_first=True)
  (fc): Linear(in_features=50, out_features=1, bias=True)
  (tanh): Tanh()
)
32651


## Power Of Choice

### C=20

In [23]:
%%time
lstm_K5_avg = copy.deepcopy(lstm)
outputs_avg, targets_avg, loss_avg, smape_avg, mae_avg, rmse_avg = poc(
    global_model = lstm_K5_avg,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    powd = n_clients,
    num_clients_per_round = num_clients_per_round,
    ratio = data_ratios
)


Starting  round 0
Clients:  [19 12  6 13  2  9 18  3  8  7 16  0 17  4 10 14  5 11  1 15]
Epoch [1/1], Train Loss: 0.046910721010395444
Epoch [1/1], Train Loss: 0.04897771403193475
Epoch [1/1], Train Loss: 0.05245149128937296
Epoch [1/1], Train Loss: 0.05329362728766034
Epoch [1/1], Train Loss: 0.05164907390384802
Epoch [1/1], Train Loss: 0.04412048386542924
Epoch [1/1], Train Loss: 0.04365116108341938
Epoch [1/1], Train Loss: 0.04820305943888214
Epoch [1/1], Train Loss: 0.051689440251461084
Epoch [1/1], Train Loss: 0.05078295644904885
Epoch [1/1], Train Loss: 0.050462761261899546
Epoch [1/1], Train Loss: 0.04799290942693394
Epoch [1/1], Train Loss: 0.0512103882086064
Epoch [1/1], Train Loss: 0.05219149782455392
Epoch [1/1], Train Loss: 0.04056534981022455
Epoch [1/1], Train Loss: 0.04887790003392314
Epoch [1/1], Train Loss: 0.050585298433101625
Epoch [1/1], Train Loss: 0.044738334670130685
Epoch [1/1], Train Loss: 0.05094433479410198
Epoch [1/1], Train Loss: 0.051968647832317014
roun

In [24]:
np.save(f'metrics/poc_outputs_C{n_clients}.npy', np.array(outputs_avg, dtype=object))
np.save(f'metrics/poc_targets_C{n_clients}.npy', np.array(targets_avg, dtype=object))
np.save(f'metrics/poc_loss_C{n_clients}.npy', loss_avg)
np.save(f'metrics/poc_smape_C{n_clients}.npy', smape_avg)
np.save(f'metrics/poc_mae_C{n_clients}.npy', mae_avg)
np.save(f'metrics/poc_rmse_C{n_clients}.npy', rmse_avg)

### C=30

In [10]:
%%time
lstm_K5_avg = copy.deepcopy(lstm)
outputs_avg, targets_avg, loss_avg, smape_avg, mae_avg, rmse_avg = poc(
    global_model = lstm_K5_avg,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    powd = n_clients,
    num_clients_per_round = num_clients_per_round,
    ratio = data_ratios
)


Starting  round 0
Clients:  [ 2 28 13 10 26 24 27 11 17 22  5 16  8 14 23 20  1 29  6  4 18 19  9  7
 25  3  0 21 15 12]
Epoch [1/1], Train Loss: 0.05716438012729797
Epoch [1/1], Train Loss: 0.054014901464272826
Epoch [1/1], Train Loss: 0.05903737747576086
Epoch [1/1], Train Loss: 0.04913039202801884
Epoch [1/1], Train Loss: 0.05644432535128933
Epoch [1/1], Train Loss: 0.05417479411698877
Epoch [1/1], Train Loss: 0.05754135998098979
Epoch [1/1], Train Loss: 0.056399599083566246
Epoch [1/1], Train Loss: 0.05176902904973499
Epoch [1/1], Train Loss: 0.05908431724778244
Epoch [1/1], Train Loss: 0.05644032152901802
Epoch [1/1], Train Loss: 0.05597252466915442
Epoch [1/1], Train Loss: 0.05808383815123567
Epoch [1/1], Train Loss: 0.056678851534213336
Epoch [1/1], Train Loss: 0.060611646921773045
Epoch [1/1], Train Loss: 0.05671954251426671
Epoch [1/1], Train Loss: 0.05683638820690768
Epoch [1/1], Train Loss: 0.0569925708031016
Epoch [1/1], Train Loss: 0.058560685786817744
Epoch [1/1], Train 

In [11]:
np.save(f'metrics/poc_outputs_C{n_clients}.npy', np.array(outputs_avg, dtype=object))
np.save(f'metrics/poc_targets_C{n_clients}.npy', np.array(targets_avg, dtype=object))
np.save(f'metrics/poc_loss_C{n_clients}.npy', loss_avg)
np.save(f'metrics/poc_smape_C{n_clients}.npy', smape_avg)
np.save(f'metrics/poc_mae_C{n_clients}.npy', mae_avg)
np.save(f'metrics/poc_rmse_C{n_clients}.npy', rmse_avg)

### C=40

In [10]:
%%time
lstm_K5_avg = copy.deepcopy(lstm)
outputs_avg, targets_avg, loss_avg, smape_avg, mae_avg, rmse_avg = poc(
    global_model = lstm_K5_avg,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    powd = n_clients,
    num_clients_per_round = num_clients_per_round,
    ratio = data_ratios
)


Starting  round 0
Clients:  [22 20 25  4 10 15 28 11 18 29 27 35 37  2 39 30 34 16 36  8 13  5 17 14
 33  7 32  1 26 12 31 24  6 23 21 19  9 38  3  0]
Epoch [1/1], Train Loss: 0.11998854916808858
Epoch [1/1], Train Loss: 0.12034852909190315
Epoch [1/1], Train Loss: 0.11373357933812907
Epoch [1/1], Train Loss: 0.11488380871846204
Epoch [1/1], Train Loss: 0.11415784287133383
Epoch [1/1], Train Loss: 0.11033946002966592
Epoch [1/1], Train Loss: 0.11247253956805378
Epoch [1/1], Train Loss: 0.13150986111057655
Epoch [1/1], Train Loss: 0.11934174337823474
Epoch [1/1], Train Loss: 0.10871749757123844
Epoch [1/1], Train Loss: 0.11409312579780814
Epoch [1/1], Train Loss: 0.11720447947404213
Epoch [1/1], Train Loss: 0.11309140734374525
Epoch [1/1], Train Loss: 0.11199901532381772
Epoch [1/1], Train Loss: 0.12660916841455866
Epoch [1/1], Train Loss: 0.11390262203557146
Epoch [1/1], Train Loss: 0.1066608228055494
Epoch [1/1], Train Loss: 0.10971812651093515
Epoch [1/1], Train Loss: 0.111335667249

In [11]:
np.save(f'metrics/poc_outputs_C{n_clients}.npy', np.array(outputs_avg, dtype=object))
np.save(f'metrics/poc_targets_C{n_clients}.npy', np.array(targets_avg, dtype=object))
np.save(f'metrics/poc_loss_C{n_clients}.npy', loss_avg)
np.save(f'metrics/poc_smape_C{n_clients}.npy', smape_avg)
np.save(f'metrics/poc_mae_C{n_clients}.npy', mae_avg)
np.save(f'metrics/poc_rmse_C{n_clients}.npy', rmse_avg)

### C=50

In [10]:
%%time
lstm_K5_avg = copy.deepcopy(lstm)
outputs_avg, targets_avg, loss_avg, smape_avg, mae_avg, rmse_avg = poc(
    global_model = lstm_K5_avg,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    powd = n_clients,
    num_clients_per_round = num_clients_per_round,
    ratio = data_ratios
)


Starting  round 0
Clients:  [28 11 10 41  2 27 38 31 22  4 33 35 26 34 18  7 14 45 48 29 15 30 32 16
 42 20 43  8 13 25  5 17 40 49  1 12 37 24  6 23 36 21 19  9 39 46  3  0
 47 44]
Epoch [1/1], Train Loss: 0.06219905668071338
Epoch [1/1], Train Loss: 0.06521609272541745
Epoch [1/1], Train Loss: 0.05596245981619827
Epoch [1/1], Train Loss: 0.06297183708686914
Epoch [1/1], Train Loss: 0.06279248418286444
Epoch [1/1], Train Loss: 0.06307858494775635
Epoch [1/1], Train Loss: 0.06476663232647946
Epoch [1/1], Train Loss: 0.056549232436477054
Epoch [1/1], Train Loss: 0.0654118150539164
Epoch [1/1], Train Loss: 0.06412319260250243
Epoch [1/1], Train Loss: 0.07328253931232862
Epoch [1/1], Train Loss: 0.059596934488841465
Epoch [1/1], Train Loss: 0.061781082468639524
Epoch [1/1], Train Loss: 0.05921757806624685
Epoch [1/1], Train Loss: 0.0665126931188362
Epoch [1/1], Train Loss: 0.06193102736558235
Epoch [1/1], Train Loss: 0.062319095512585965
Epoch [1/1], Train Loss: 0.06472930774491814
Epoch

In [11]:
np.save(f'metrics/poc_outputs_C{n_clients}.npy', np.array(outputs_avg, dtype=object))
np.save(f'metrics/poc_targets_C{n_clients}.npy', np.array(targets_avg, dtype=object))
np.save(f'metrics/poc_loss_C{n_clients}.npy', loss_avg)
np.save(f'metrics/poc_smape_C{n_clients}.npy', smape_avg)
np.save(f'metrics/poc_mae_C{n_clients}.npy', mae_avg)
np.save(f'metrics/poc_rmse_C{n_clients}.npy', rmse_avg)

### C=60

In [10]:
%%time
lstm_K5_avg = copy.deepcopy(lstm)
outputs_avg, targets_avg, loss_avg, smape_avg, mae_avg, rmse_avg = poc(
    global_model = lstm_K5_avg,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    powd = n_clients,
    num_clients_per_round = num_clients_per_round,
    ratio = data_ratios
)


Starting  round 0
Clients:  [26 35 59 28 11  2 34 58 40 22  4 10 30 41 33 43 49  7 14 32 50 29 42 54
 18 56 27 15  5 31 16 51 20 52  8 13 25 37 17 45 48 57 38  1 12 46 24  6
 23 36 21 19  9 39 55  3  0 53 47 44]
Epoch [1/1], Train Loss: 0.07342777181682847
Epoch [1/1], Train Loss: 0.07074065393369112
Epoch [1/1], Train Loss: 0.07947407536474722
Epoch [1/1], Train Loss: 0.07385781058110295
Epoch [1/1], Train Loss: 0.08100234038595643
Epoch [1/1], Train Loss: 0.07410670135037176
Epoch [1/1], Train Loss: 0.06989926937967542
Epoch [1/1], Train Loss: 0.07624194351956247
Epoch [1/1], Train Loss: 0.07511386434946742
Epoch [1/1], Train Loss: 0.0781613117017384
Epoch [1/1], Train Loss: 0.07596149155870079
Epoch [1/1], Train Loss: 0.06945052636521203
Epoch [1/1], Train Loss: 0.07474095154819745
Epoch [1/1], Train Loss: 0.07472238374508118
Epoch [1/1], Train Loss: 0.09115039171384909
Epoch [1/1], Train Loss: 0.07476877208266938
Epoch [1/1], Train Loss: 0.07310754037462171
Epoch [1/1], Train Loss

In [11]:
np.save(f'metrics/poc_outputs_C{n_clients}.npy', np.array(outputs_avg, dtype=object))
np.save(f'metrics/poc_targets_C{n_clients}.npy', np.array(targets_avg, dtype=object))
np.save(f'metrics/poc_loss_C{n_clients}.npy', loss_avg)
np.save(f'metrics/poc_smape_C{n_clients}.npy', smape_avg)
np.save(f'metrics/poc_mae_C{n_clients}.npy', mae_avg)
np.save(f'metrics/poc_rmse_C{n_clients}.npy', rmse_avg)

### C=70

In [10]:
%%time
lstm_K5_avg = copy.deepcopy(lstm)
outputs_avg, targets_avg, loss_avg, smape_avg, mae_avg, rmse_avg = poc(
    global_model = lstm_K5_avg,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    powd = n_clients,
    num_clients_per_round = num_clients_per_round,
    ratio = data_ratios
)


Starting  round 0
Clients:  [26 27 48 22 30 51  7 59 34 69 56 28 31 42 33 55 68 62 43  4 65 50  2 40
 11  3 54 45 10 41 49 53 57 32 14 67 19 29 52 35 18  0 15  5 16 20 66  8
 13 25 37 17 60 46 63 39 38  1 58 12 61 24  6 23 36 21  9 64 47 44]
Epoch [1/1], Train Loss: 0.10873676024909527
Epoch [1/1], Train Loss: 0.11097578963797006
Epoch [1/1], Train Loss: 0.10526498420430079
Epoch [1/1], Train Loss: 0.11678911206711613
Epoch [1/1], Train Loss: 0.11068597768566438
Epoch [1/1], Train Loss: 0.11193024486835514
Epoch [1/1], Train Loss: 0.10799681248941592
Epoch [1/1], Train Loss: 0.11431689287668893
Epoch [1/1], Train Loss: 0.10377998330763405
Epoch [1/1], Train Loss: 0.1233708444051445
Epoch [1/1], Train Loss: 0.10828834512670128
Epoch [1/1], Train Loss: 0.10939836329115285
Epoch [1/1], Train Loss: 0.10206973632531506
Epoch [1/1], Train Loss: 0.1077523898399834
Epoch [1/1], Train Loss: 0.14241786141480717
Epoch [1/1], Train Loss: 0.10701519862881727
Epoch [1/1], Train Loss: 0.120713859131

In [11]:
np.save(f'metrics/poc_outputs_C{n_clients}.npy', np.array(outputs_avg, dtype=object))
np.save(f'metrics/poc_targets_C{n_clients}.npy', np.array(targets_avg, dtype=object))
np.save(f'metrics/poc_loss_C{n_clients}.npy', loss_avg)
np.save(f'metrics/poc_smape_C{n_clients}.npy', smape_avg)
np.save(f'metrics/poc_mae_C{n_clients}.npy', mae_avg)
np.save(f'metrics/poc_rmse_C{n_clients}.npy', rmse_avg)

### C=80

In [10]:
%%time
lstm_K5_avg = copy.deepcopy(lstm)
outputs_avg, targets_avg, loss_avg, smape_avg, mae_avg, rmse_avg = poc(
    global_model = lstm_K5_avg,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    powd = n_clients,
    num_clients_per_round = num_clients_per_round,
    ratio = data_ratios
)


Starting  round 0
Clients:  [50 27 30 77 45 71 66 68 48 69 40 63  2 72 52 34  3 22  7 26 42 38 41 79
 55 17 60 62  6 56 73  4 33 53 28 11 57 23 10 31 43 61  1 32 75 14 54 19
 29 49 24 35 18  0 78 15  5 59 16 51 20 74  8 13 25 37 46 39 65 58 12 70
 36 21  9 76 67 64 47 44]
Epoch [1/1], Train Loss: 0.07548380367058728
Epoch [1/1], Train Loss: 0.07179364482206957
Epoch [1/1], Train Loss: 0.0712720787352217
Epoch [1/1], Train Loss: 0.07151988833876595
Epoch [1/1], Train Loss: 0.07383934272054055
Epoch [1/1], Train Loss: 0.0699852342118642
Epoch [1/1], Train Loss: 0.0711661510369075
Epoch [1/1], Train Loss: 0.06993040562208208
Epoch [1/1], Train Loss: 0.06727666035294534
Epoch [1/1], Train Loss: 0.07080177922866177
Epoch [1/1], Train Loss: 0.07194673075407211
Epoch [1/1], Train Loss: 0.0670341827707099
Epoch [1/1], Train Loss: 0.07083939802500286
Epoch [1/1], Train Loss: 0.06899983934792023
Epoch [1/1], Train Loss: 0.07209740026987024
Epoch [1/1], Train Loss: 0.0669409692553537
Epoch [1/1]

In [11]:
np.save(f'metrics/poc_outputs_C{n_clients}.npy', np.array(outputs_avg, dtype=object))
np.save(f'metrics/poc_targets_C{n_clients}.npy', np.array(targets_avg, dtype=object))
np.save(f'metrics/poc_loss_C{n_clients}.npy', loss_avg)
np.save(f'metrics/poc_smape_C{n_clients}.npy', smape_avg)
np.save(f'metrics/poc_mae_C{n_clients}.npy', mae_avg)
np.save(f'metrics/poc_rmse_C{n_clients}.npy', rmse_avg)

### C=90

In [10]:
%%time
lstm_K5_avg = copy.deepcopy(lstm)
outputs_avg, targets_avg, loss_avg, smape_avg, mae_avg, rmse_avg = poc(
    global_model = lstm_K5_avg,
    client_train_loader = train_loader,
    test_loader = test_loader,
    label_sc = label_scalers,
    n_clients = n_clients,
    batch_size = batch_size,
    num_local_epochs = num_local_epochs,
    lr = lr,
    max_rounds = max_rounds,
    model_type = model_type,
    device = device,
    powd = n_clients,
    num_clients_per_round = num_clients_per_round,
    ratio = data_ratios
)


Starting  round 0
Clients:  [ 2 13 53 41 66 30 45 43 78 89  7 26 33 63  8 16 24 56 77 42 22  6 61 48
 80 54 73 79 82  3 62 75 27 18 50 51 74 59 55  4 15 17 40 38  5 81 68  0
 34 28 11 35 23 52 10 31 60 57 71  1 32 76 14 85 19 29 49 87 88 69 20 84
 72 25 37 46 39 65 58 12 70 36 21 83  9 86 67 64 47 44]
Epoch [1/1], Train Loss: 0.07064124605884509
Epoch [1/1], Train Loss: 0.07361537364444565
Epoch [1/1], Train Loss: 0.07680228884731019
Epoch [1/1], Train Loss: 0.0716584183807884
Epoch [1/1], Train Loss: 0.07140715530009142
Epoch [1/1], Train Loss: 0.07148105854035489
Epoch [1/1], Train Loss: 0.07420653054889824
Epoch [1/1], Train Loss: 0.07200133910269607
Epoch [1/1], Train Loss: 0.07358137844130397
Epoch [1/1], Train Loss: 0.07750650987561261
Epoch [1/1], Train Loss: 0.07035187099661146
Epoch [1/1], Train Loss: 0.07037175686231681
Epoch [1/1], Train Loss: 0.0898557331945215
Epoch [1/1], Train Loss: 0.06735794039975322
Epoch [1/1], Train Loss: 0.07241643985201207
Epoch [1/1], Train Loss

In [11]:
np.save(f'metrics/poc_outputs_C{n_clients}.npy', np.array(outputs_avg, dtype=object))
np.save(f'metrics/poc_targets_C{n_clients}.npy', np.array(targets_avg, dtype=object))
np.save(f'metrics/poc_loss_C{n_clients}.npy', loss_avg)
np.save(f'metrics/poc_smape_C{n_clients}.npy', smape_avg)
np.save(f'metrics/poc_mae_C{n_clients}.npy', mae_avg)
np.save(f'metrics/poc_rmse_C{n_clients}.npy', rmse_avg)